# imports

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# pdf ingestion

In [ ]:
loader = PyPDFLoader("../data/raw/attention_is_all_you_need.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)
print(len(chunks))

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)
print(len(chunks))

52


In [16]:
print(chunks[0].page_content)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions


In [22]:
chunks[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/raw/attention_is_all_you_need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kai

In [28]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 744.59it/s]


In [30]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../vector_db/chroma"
)

In [31]:
results = vector_db.similarity_search(
    "What is self-attention?",
    k=3
)

In [32]:
results

[Document(metadata={'moddate': '2024-04-10T21:11:43+00:00', 'subject': '', 'title': '', 'page_label': '3', 'creationdate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'page': 2, 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'trapped': '/False', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'author': '', 'source': '../data/raw/attention_is_all_you_need.pdf'}, page_content='3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'),
 Document(metadata={'subject': '', 'source': '../data/raw/attention_is_all_you_need.pdf', 'trapped': '/False', 'author': '', 'page': 1, 'creationdate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'keywords': '', 'moddate': '2024-04-1

In [33]:
context = "\n\n".join(
    [result.page_content for result in results]
)

In [39]:
query = "What is self-attention?"

In [40]:
prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""

In [41]:
import ollama

response = ollama.chat(
    model="phi3",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [42]:
response

ChatResponse(model='phi3', created_at='2026-05-24T10:59:01.901805005Z', done=True, done_reason='stop', total_duration=9397249661, load_duration=19435375, prompt_eval_count=539, prompt_eval_duration=5196904227, eval_count=77, eval_duration=4179147209, message=Message(role='assistant', content='Self-attention, sometimes referred to as intra-attention, is an attention mechanism employed in mapping different positions within a single sequence for representation computation purposes. It has shown successful application across various tasks such as reading comprehension, abstractive summarization and learning task-independent sentence representations among others [4, 27, 28, 22].', thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [44]:
response['message']['content']

'Self-attention, sometimes referred to as intra-attention, is an attention mechanism employed in mapping different positions within a single sequence for representation computation purposes. It has shown successful application across various tasks such as reading comprehension, abstractive summarization and learning task-independent sentence representations among others [4, 27, 28, 22].'